In [7]:
import os
from google.colab import userdata
import torch.optim as optim
import wandb
import sys
from dataset import FERDataset, load_data, EMOTION_LABELS
from torch.utils.data import DataLoader
import torch.nn as nn
import torch.nn.functional as F
import torch


In [1]:
import os
from google.colab import userdata

os.environ['WANDB_API_KEY'] = userdata.get('WANDB_API_KEY')
os.environ['KAGGLE_KEY'] = userdata.get('KAGGLE_KEY')
os.environ['GITHUB_TOKEN'] = userdata.get('GITHUB_TOKEN')
os.environ['GITHUB_USERNAME'] = userdata.get('GITHUB_USERNAME')

!git clone https://{os.environ['GITHUB_TOKEN']}@github.com/{os.environ['GITHUB_USERNAME']}/ML_hw_04_Facial-Expression-Recognition-Challenge.git
%cd ML_hw_04_Facial-Expression-Recognition-Challenge

!pip install -q wandb kaggle

import wandb
wandb.login(key=os.environ['WANDB_API_KEY'])

os.makedirs('/root/.kaggle', exist_ok=True)
with open('/root/.kaggle/access_token', 'w') as f:
    f.write(os.environ['KAGGLE_KEY'])

os.makedirs('/content/data', exist_ok=True)
!kaggle competitions download -c challenges-in-representation-learning-facial-expression-recognition-challenge -p /content/data
!unzip -q -o /content/data/*.zip -d /content/data

import sys
sys.path.append('/content/ML_hw_04_Facial-Expression-Recognition-Challenge/src')
from dataset import FERDataset, load_data, EMOTION_LABELS

print("Setup complete!")

Cloning into 'ML_hw_04_Facial-Expression-Recognition-Challenge'...
remote: Enumerating objects: 41, done.
remote: Counting objects: 100% (41/41), done.
remote: Compressing objects: 100% (33/33), done.
remote: Total 41 (delta 14), reused 10 (delta 2), pack-reused 0 (from 0)
Receiving objects: 100% (41/41), 87.75 KiB | 5.48 MiB/s, done.
Resolving deltas: 100% (14/14), done.
/content/ML_hw_04_Facial-Expression-Recognition-Challenge


/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: [wandb.login()] Using explicit session credentials for https://api.wandb.ai.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: kgord23 (kgord23-free-university-of-tbilisi-) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


100% 285M/285M [00:04<00:00, 65.0MB/s]

Setup complete!


In [2]:
from torch.utils.data import DataLoader

train, val, test=load_data('/content/data/train.csv')
print("Train:", train.shape, "Val:", val.shape, "Test:", test.shape)

train_data=FERDataset(train)
val_data=FERDataset(val)
test_data=FERDataset(test)

BATCH_SIZE=64
train_loader = DataLoader(train_data, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_data, batch_size=BATCH_SIZE, shuffle=False)
test_loader = DataLoader(test_data, batch_size=BATCH_SIZE, shuffle=False)

print("Train batches:", len(train_loader))
print("Val batches:", len(val_loader))
print("Test batches:", len(test_loader))

Train: (20095, 2) Val: (4307, 2) Test: (4307, 2)
Train batches: 314
Val batches: 68
Test batches: 68


In [3]:
import torch.nn as nn
import torch.nn.functional as F

class SimpleCNN(nn.Module):
  def __init__(self, num_classes=7):
      super().__init__()
      self.conv1 = nn.Conv2d(1, 16, kernel_size=3, padding=1)
      self.conv2 = nn.Conv2d(16, 32, kernel_size=3, padding=1)
      self.pool = nn.MaxPool2d(2, 2)
      self.fc = nn.Linear(32 * 12 * 12, num_classes)
  def forward(self, x):
      x = self.pool(F.relu(self.conv1(x)))  # (16, 24, 24)
      x = self.pool(F.relu(self.conv2(x)))  # (32, 12, 12)
      x = x.view(x.size(0), -1)             # flatten
      x = self.fc(x)
      return x

In [4]:
# Quick test - check forward pass shapes
model = SimpleCNN()
sample_batch, _ = next(iter(train_loader))
print("Input shape:", sample_batch.shape)
output = model(sample_batch)
print("Output shape:", output.shape)  # should be (batch_size, 7)

Input shape: torch.Size([64, 1, 48, 48])
Output shape: torch.Size([64, 7])


In [6]:
import torch
criterion=nn.CrossEntropyLoss()
sample_batch, sample_labels=next(iter(train_loader))
output=model(sample_batch)
loss=criterion(output, sample_labels)
print("Loss:", loss.item())
print("Expected (untrained, ~uniform):", torch.log(torch.tensor(7.0)).item())

Loss: 1.9696006774902344
Expected (untrained, ~uniform): 1.9459102153778076


In [8]:
def train_model_buggy(model, train_loader, val_loader, config, group_name, run_name):
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    model = model.to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=config['learning_rate'])

    run = wandb.init(project="facial-expression-recognition", group=group_name, name=run_name, config=config)

    for epoch in range(config['epochs']):
        model.train()
        train_loss, train_correct, train_total = 0, 0, 0
        for images, labels in train_loader:
            images, labels = images.to(device), labels.to(device)
            # optimizer.zero_grad()  <-- INTENTIONALLY OMITTED
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            train_loss += loss.item() * images.size(0)
            _, predicted = torch.max(outputs, 1)
            train_correct += (predicted == labels).sum().item()
            train_total += labels.size(0)

        train_loss /= train_total
        train_acc = train_correct / train_total
        wandb.log({"epoch": epoch+1, "train_loss": train_loss, "train_acc": train_acc})
        print(f"Epoch {epoch+1}: Loss={train_loss:.4f}, Acc={train_acc:.4f}")

    wandb.finish()
    return model

In [9]:
config_bug = {"architecture": "SimpleCNN-no-zero-grad", "learning_rate": 0.001, "batch_size": 64, "epochs": 5, "optimizer": "Adam"}
model_bug = SimpleCNN()
model_bug = train_model_buggy(model_bug, train_loader, val_loader, config_bug, group_name="sanity-checks", run_name="no-zero-grad-bug-demo")

Epoch 1: Loss=1.8866, Acc=0.2082
Epoch 2: Loss=1.8360, Acc=0.2514
Epoch 3: Loss=1.8144, Acc=0.2514
Epoch 4: Loss=1.8108, Acc=0.2514
Epoch 5: Loss=1.8153, Acc=0.2514


epoch,▁▃▅▆█
train_acc,▁████
train_loss,█▃▁▁▁
epoch,5
train_acc,0.25136
train_loss,1.81532


In [13]:
def train_model(model, train_loader, val_loader, config, group_name, run_name):
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    model = model.to(device)

    criterion = nn.CrossEntropyLoss()
    if config.get('optimizer', 'Adam')=='SGD':
      optimizer = optim.SGD(model.parameters(), lr=config['learning_rate'], momentum=0.9)
    else:
      optimizer = optim.Adam(model.parameters(), lr=config['learning_rate'])

    run = wandb.init(
        project="facial-expression-recognition",
        group=group_name,
        name=run_name,
        config=config
    )

    for epoch in range(config['epochs']):
        # --- Training ---
        model.train()
        train_loss, train_correct, train_total = 0, 0, 0
        for images, labels in train_loader:
            images, labels = images.to(device), labels.to(device)

            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            train_loss += loss.item() * images.size(0)
            _, predicted = torch.max(outputs, 1)
            train_correct += (predicted == labels).sum().item()
            train_total += labels.size(0)

        train_loss /= train_total
        train_acc = train_correct / train_total

        # --- Validation ---
        model.eval()
        val_loss, val_correct, val_total = 0, 0, 0
        with torch.no_grad():
            for images, labels in val_loader:
                images, labels = images.to(device), labels.to(device)
                outputs = model(images)
                loss = criterion(outputs, labels)

                val_loss += loss.item() * images.size(0)
                _, predicted = torch.max(outputs, 1)
                val_correct += (predicted == labels).sum().item()
                val_total += labels.size(0)

        val_loss /= val_total
        val_acc = val_correct / val_total


        wandb.log({
            "epoch": epoch + 1,
            "train_loss": train_loss,
            "train_acc": train_acc,
            "val_loss": val_loss,
            "val_acc": val_acc
        })

        print(f"Epoch {epoch+1}/{config['epochs']} | "
              f"Train Loss: {train_loss:.4f}, Train Acc: {train_acc:.4f} | "
              f"Val Loss: {val_loss:.4f}, Val Acc: {val_acc:.4f}")

    wandb.finish()
    return model

In [11]:
small_subset = train.sample(n=200, random_state=42)
small_dataset = FERDataset(small_subset)
small_loader = DataLoader(small_dataset, batch_size=32, shuffle=True)

config_sanity = {
    "architecture": "SimpleCNN-sanity",
    "learning_rate": 0.001,
    "batch_size": 32,
    "epochs": 30,
    "optimizer": "Adam"
}

model_sanity = SimpleCNN()
model_sanity = train_model(model_sanity, small_loader, small_loader, config_sanity,
                            group_name="sanity-checks",
                            run_name="overfit-200-samples")

Epoch 1/30 | Train Loss: 1.8745, Train Acc: 0.2300 | Val Loss: 1.7882, Val Acc: 0.2600
Epoch 2/30 | Train Loss: 1.8250, Train Acc: 0.1800 | Val Loss: 1.7912, Val Acc: 0.2650
Epoch 3/30 | Train Loss: 1.7925, Train Acc: 0.2700 | Val Loss: 1.7539, Val Acc: 0.2700
Epoch 4/30 | Train Loss: 1.7512, Train Acc: 0.2700 | Val Loss: 1.7143, Val Acc: 0.2650
Epoch 5/30 | Train Loss: 1.7053, Train Acc: 0.2700 | Val Loss: 1.6584, Val Acc: 0.3800
Epoch 6/30 | Train Loss: 1.6600, Train Acc: 0.3650 | Val Loss: 1.6000, Val Acc: 0.4300
Epoch 7/30 | Train Loss: 1.5971, Train Acc: 0.3950 | Val Loss: 1.5373, Val Acc: 0.4500
Epoch 8/30 | Train Loss: 1.5630, Train Acc: 0.3950 | Val Loss: 1.4844, Val Acc: 0.4450
Epoch 9/30 | Train Loss: 1.5093, Train Acc: 0.4400 | Val Loss: 1.4373, Val Acc: 0.4600
Epoch 10/30 | Train Loss: 1.4201, Train Acc: 0.5300 | Val Loss: 1.3643, Val Acc: 0.5450
Epoch 11/30 | Train Loss: 1.3656, Train Acc: 0.5150 | Val Loss: 1.2789, Val Acc: 0.5600
Epoch 12/30 | Train Loss: 1.2922, Train A

epoch,▁▁▁▂▂▂▂▃▃▃▃▄▄▄▄▅▅▅▅▆▆▆▆▇▇▇▇███
train_acc,▁▁▂▂▂▃▃▃▄▄▄▅▅▅▆▆▆▆▆▇▇▇▇▇▇▇████
train_loss,███▇▇▇▇▇▆▆▆▅▅▅▄▄▄▄▃▃▃▂▂▂▂▂▂▁▁▁
val_acc,▁▁▁▁▂▃▃▃▃▄▄▄▅▅▅▅▆▆▆▆▇▇▇▇▇█████
val_loss,████▇▇▇▇▆▆▆▅▅▄▄▄▄▃▃▃▃▂▂▂▂▂▁▁▁▁
epoch,30
train_acc,0.875
train_loss,0.40713
val_acc,0.925
val_loss,0.34116


In [12]:
config1 = {
    "architecture": "SimpleCNN",
    "learning_rate": 0.001,
    "batch_size": 64,
    "epochs": 10,
    "optimizer": "Adam"
}

model1 = SimpleCNN()
model1 = train_model(model1, train_loader, val_loader, config1,
                      group_name="architecture-1-simple-cnn",
                      run_name="lr0.001-bs64")

Epoch 1/10 | Train Loss: 1.6918, Train Acc: 0.3293 | Val Loss: 1.6101, Val Acc: 0.3638
Epoch 2/10 | Train Loss: 1.5519, Train Acc: 0.4050 | Val Loss: 1.5250, Val Acc: 0.4142
Epoch 3/10 | Train Loss: 1.4674, Train Acc: 0.4419 | Val Loss: 1.4874, Val Acc: 0.4370
Epoch 4/10 | Train Loss: 1.4005, Train Acc: 0.4725 | Val Loss: 1.4155, Val Acc: 0.4588
Epoch 5/10 | Train Loss: 1.3444, Train Acc: 0.4951 | Val Loss: 1.4027, Val Acc: 0.4711
Epoch 6/10 | Train Loss: 1.3000, Train Acc: 0.5133 | Val Loss: 1.3866, Val Acc: 0.4695
Epoch 7/10 | Train Loss: 1.2600, Train Acc: 0.5279 | Val Loss: 1.3897, Val Acc: 0.4669
Epoch 8/10 | Train Loss: 1.2282, Train Acc: 0.5426 | Val Loss: 1.3663, Val Acc: 0.4834
Epoch 9/10 | Train Loss: 1.1904, Train Acc: 0.5585 | Val Loss: 1.3642, Val Acc: 0.4925
Epoch 10/10 | Train Loss: 1.1627, Train Acc: 0.5720 | Val Loss: 1.3666, Val Acc: 0.4957


epoch,▁▂▃▃▄▅▆▆▇█
train_acc,▁▃▄▅▆▆▇▇██
train_loss,█▆▅▄▃▃▂▂▁▁
val_acc,▁▄▅▆▇▇▆▇██
val_loss,█▆▅▂▂▂▂▁▁▁
epoch,10
train_acc,0.57198
train_loss,1.16267
val_acc,0.4957
val_loss,1.36659


In [14]:
config2 = {
    "architecture": "SimpleCNN",
    "learning_rate": 0.0001,
    "batch_size": 64,
    "epochs": 10,
    "optimizer": "Adam"
}

model2 = SimpleCNN()
model2 = train_model(model2, train_loader, val_loader, config2,
                      group_name="architecture-1-simple-cnn",
                      run_name="lr0.0001-bs64-adam")

Epoch 1/10 | Train Loss: 1.7875, Train Acc: 0.2614 | Val Loss: 1.7442, Val Acc: 0.2823
Epoch 2/10 | Train Loss: 1.7171, Train Acc: 0.3082 | Val Loss: 1.6894, Val Acc: 0.3297
Epoch 3/10 | Train Loss: 1.6611, Train Acc: 0.3520 | Val Loss: 1.6451, Val Acc: 0.3615
Epoch 4/10 | Train Loss: 1.6234, Train Acc: 0.3741 | Val Loss: 1.6170, Val Acc: 0.3884
Epoch 5/10 | Train Loss: 1.5959, Train Acc: 0.3878 | Val Loss: 1.5938, Val Acc: 0.3917
Epoch 6/10 | Train Loss: 1.5758, Train Acc: 0.3985 | Val Loss: 1.5886, Val Acc: 0.3963
Epoch 7/10 | Train Loss: 1.5585, Train Acc: 0.4076 | Val Loss: 1.5685, Val Acc: 0.3931
Epoch 8/10 | Train Loss: 1.5461, Train Acc: 0.4131 | Val Loss: 1.5579, Val Acc: 0.4024
Epoch 9/10 | Train Loss: 1.5327, Train Acc: 0.4205 | Val Loss: 1.5487, Val Acc: 0.4165
Epoch 10/10 | Train Loss: 1.5224, Train Acc: 0.4245 | Val Loss: 1.5337, Val Acc: 0.4189


epoch,▁▂▃▃▄▅▆▆▇█
train_acc,▁▃▅▆▆▇▇███
train_loss,█▆▅▄▃▂▂▂▁▁
val_acc,▁▃▅▆▇▇▇▇██
val_loss,█▆▅▄▃▃▂▂▁▁
epoch,10
train_acc,0.42448
train_loss,1.52244
val_acc,0.41885
val_loss,1.53367


In [15]:
train_loader_bs32 = DataLoader(train_data, batch_size=32, shuffle=True)
val_loader_bs32 = DataLoader(val_data, batch_size=32, shuffle=False)

config3 = {"architecture": "SimpleCNN", "learning_rate": 0.001, "batch_size": 32, "epochs": 10, "optimizer": "Adam"}
model3 = SimpleCNN()
model3 = train_model(model3, train_loader_bs32, val_loader_bs32, config3, group_name="architecture-1-simple-cnn", run_name="lr0.001-bs32-adam")

Epoch 1/10 | Train Loss: 1.6764, Train Acc: 0.3342 | Val Loss: 1.5672, Val Acc: 0.3947
Epoch 2/10 | Train Loss: 1.5167, Train Acc: 0.4192 | Val Loss: 1.4653, Val Acc: 0.4467
Epoch 3/10 | Train Loss: 1.4132, Train Acc: 0.4625 | Val Loss: 1.4118, Val Acc: 0.4746
Epoch 4/10 | Train Loss: 1.3538, Train Acc: 0.4870 | Val Loss: 1.3889, Val Acc: 0.4783
Epoch 5/10 | Train Loss: 1.3037, Train Acc: 0.5111 | Val Loss: 1.4061, Val Acc: 0.4664
Epoch 6/10 | Train Loss: 1.2663, Train Acc: 0.5212 | Val Loss: 1.3484, Val Acc: 0.4929
Epoch 7/10 | Train Loss: 1.2289, Train Acc: 0.5406 | Val Loss: 1.3585, Val Acc: 0.4829
Epoch 8/10 | Train Loss: 1.1930, Train Acc: 0.5551 | Val Loss: 1.3509, Val Acc: 0.4855
Epoch 9/10 | Train Loss: 1.1600, Train Acc: 0.5685 | Val Loss: 1.3423, Val Acc: 0.5003
Epoch 10/10 | Train Loss: 1.1292, Train Acc: 0.5793 | Val Loss: 1.3614, Val Acc: 0.4915


epoch,▁▂▃▃▄▅▆▆▇█
train_acc,▁▃▅▅▆▆▇▇██
train_loss,█▆▅▄▃▃▂▂▁▁
val_acc,▁▄▆▇▆█▇▇█▇
val_loss,█▅▃▂▃▁▂▁▁▂
epoch,10
train_acc,0.5793
train_loss,1.12921
val_acc,0.49153
val_loss,1.36135


In [16]:
config4 = {"architecture": "SimpleCNN", "learning_rate": 0.001, "batch_size": 64, "epochs": 10, "optimizer": "SGD"}
model4 = SimpleCNN()
model4 = train_model(model4, train_loader, val_loader, config4, group_name="architecture-1-simple-cnn", run_name="lr0.001-bs64-sgd")

Epoch 1/10 | Train Loss: 1.8071, Train Acc: 0.2525 | Val Loss: 1.8057, Val Acc: 0.2431
Epoch 2/10 | Train Loss: 1.7735, Train Acc: 0.2648 | Val Loss: 1.7538, Val Acc: 0.2679
Epoch 3/10 | Train Loss: 1.7443, Train Acc: 0.2913 | Val Loss: 1.7327, Val Acc: 0.3028
Epoch 4/10 | Train Loss: 1.7142, Train Acc: 0.3124 | Val Loss: 1.7119, Val Acc: 0.3234
Epoch 5/10 | Train Loss: 1.6856, Train Acc: 0.3390 | Val Loss: 1.6915, Val Acc: 0.3309
Epoch 6/10 | Train Loss: 1.6619, Train Acc: 0.3518 | Val Loss: 1.6632, Val Acc: 0.3443
Epoch 7/10 | Train Loss: 1.6432, Train Acc: 0.3640 | Val Loss: 1.6440, Val Acc: 0.3706
Epoch 8/10 | Train Loss: 1.6240, Train Acc: 0.3733 | Val Loss: 1.6356, Val Acc: 0.3673
Epoch 9/10 | Train Loss: 1.6129, Train Acc: 0.3798 | Val Loss: 1.6174, Val Acc: 0.3743
Epoch 10/10 | Train Loss: 1.5997, Train Acc: 0.3846 | Val Loss: 1.6105, Val Acc: 0.3798


epoch,▁▂▃▃▄▅▆▆▇█
train_acc,▁▂▃▄▆▆▇▇██
train_loss,█▇▆▅▄▃▂▂▁▁
val_acc,▁▂▄▅▅▆█▇██
val_loss,█▆▅▅▄▃▂▂▁▁
epoch,10
train_acc,0.38457
train_loss,1.59972
val_acc,0.37985
val_loss,1.61045


In [17]:
config5 = {"architecture": "SimpleCNN", "learning_rate": 0.01, "batch_size": 64, "epochs": 10, "optimizer": "Adam"}
model5 = SimpleCNN()
model5 = train_model(model5, train_loader, val_loader, config5, group_name="architecture-1-simple-cnn", run_name="lr0.01-bs64-adam")

Epoch 1/10 | Train Loss: 1.7675, Train Acc: 0.2846 | Val Loss: 1.6536, Val Acc: 0.3534
Epoch 2/10 | Train Loss: 1.6377, Train Acc: 0.3608 | Val Loss: 1.6152, Val Acc: 0.3703
Epoch 3/10 | Train Loss: 1.6023, Train Acc: 0.3803 | Val Loss: 1.6269, Val Acc: 0.3703
Epoch 4/10 | Train Loss: 1.5893, Train Acc: 0.3895 | Val Loss: 1.6037, Val Acc: 0.3787
Epoch 5/10 | Train Loss: 1.5835, Train Acc: 0.3918 | Val Loss: 1.5986, Val Acc: 0.3787
Epoch 6/10 | Train Loss: 1.5768, Train Acc: 0.3926 | Val Loss: 1.6185, Val Acc: 0.3773
Epoch 7/10 | Train Loss: 1.5742, Train Acc: 0.3926 | Val Loss: 1.5910, Val Acc: 0.3884
Epoch 8/10 | Train Loss: 1.5675, Train Acc: 0.3959 | Val Loss: 1.5849, Val Acc: 0.3901
Epoch 9/10 | Train Loss: 1.5583, Train Acc: 0.3979 | Val Loss: 1.5776, Val Acc: 0.3942
Epoch 10/10 | Train Loss: 1.5464, Train Acc: 0.4087 | Val Loss: 1.5939, Val Acc: 0.3935


epoch,▁▂▃▃▄▅▆▆▇█
train_acc,▁▅▆▇▇▇▇▇▇█
train_loss,█▄▃▂▂▂▂▂▁▁
val_acc,▁▄▄▅▅▅▇▇██
val_loss,█▄▆▃▃▅▂▂▁▃
epoch,10
train_acc,0.40866
train_loss,1.54642
val_acc,0.39355
val_loss,1.59386
